# Secondary Forest Edge Removal
Creates mask to consider for analysis only the secondary forest pixels that are surrounded by other secondary forest pixels on all sides (to avoid edge effects and biomass underestimations).
Estimates area of secondary forests per 1km². This is used to make total carbon sequestration estimates/projections.

* **Imports:**
    * Collection 9 MapBiomas Secondary Vegetation Age

* **Exports:**
    * `distance_to_secondary` to GEE Image
    * `pastureland_area_1km` to GEE Image

In [2]:
import ee
import geemap
from utils import *
initialize()

config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

mapbiomas, lulc = desired_lulc()

## Remove isolated pixels

In the map, there were isolated pixels, often around the edges of forest patches. These would likely be due to misclassification, or follow different behaviors due to edge effects.

To avoid this issue, we remove all edge pixels using a distance transform approach. First, we detect the edges of secondary vegetation patches using a zero-crossing algorithm. Then, we calculate the distance from each pixel to the nearest edge pixel. Finally, we mask out all pixels within 30 meters of an edge.

This also excludes all pixels in edges of larger patches, which may have their biomass estimates affected by the neighboring non-forest land cover since ESA CCI Biomass is at 100m resolution. This is a conservative approach, but it ensures that the biomass estimates are not influenced by edge effects.

Effectively, this ensures only secondary forest pixels that are surrounded by other secondary forest pixels on all sides are included for analysis.

In [ ]:
age = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_secondary_vegetation_age_v1").select("secondary_vegetation_age_2020").clip(roi)

edge_detec = age.unmask(-1).zeroCrossing()

distance_to_secondary_edge = edge_detec.fastDistanceTransform(100, 'pixels').sqrt() \
    .multiply(ee.Image.pixelArea().sqrt()).toInt32().rename("distance_to_secondary_edge")

# export_image(distance_to_secondary_edge, "distance_to_secondary_edge", region = roi, scale = 30)

## estimate_area

- Total pasture area per 1km2

In [ ]:
def estimate_area(image, name):

    biomes = ee.Image(f"{config.data_folder}/categorical").select("biome")

    image_mask = biomes.reduceResolution(
        ee.Reducer.first(), maxPixels=65536
        ).reproject(
        crs = image.projection().getInfo()['crs'],
        crsTransform = image.projection().getInfo()['transform']
        ).updateMask(image)

    image_scale = round(image.projection().nominalScale().getInfo())
    closest_multiple = round(1000 / image_scale) * image_scale

    # First, sample locations based only on the age band
    grid = geemap.create_grid(image.geometry(), closest_multiple, image.projection())

    # Create an image representing pixel area
    area_image = ee.Image.pixelArea().updateMask(image_mask)

    # Function to compute valid area per grid cell
    def compute_area(feature):
        valid_area = area_image.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=feature.geometry(),
            crs=image.projection(),
            crsTransform=image.projection().getInfo()['transform'],
            maxPixels=1e10
        ).getNumber('area')
        return feature.set('valid_area', valid_area)

    grid_with_area = grid.map(compute_area)

    area_image_raster = grid_with_area.reduceToImage(
        properties=['valid_area'],
        reducer=ee.Reducer.first()
    )

    task = ee.batch.Export.image.toAsset(
        image = area_image_raster,
        description = f"{name}_area_1km",
        assetId = f"projects/amazon-forest-regrowth/assets/{name}_area_1km",
        crs = area_image_raster.projection().crs(),
        crsTransform=area_image_raster.projection().getInfo()['transform'],
        region = image.geometry()
    )
    task.start()


In [ ]:
pastureland = (ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1")
            .select([f"classification_{year}" for year in config.range_1985_2020])
            .byte()
            .rename([str(year) for year in config.range_1985_2020]))
pastureland = pastureland.select("2020").eq(15).unmask(0).rename("pastureland")

# estimate_area(pastureland, "pastureland")